# 05 - Error analysis

Uses the predictions of the best transformer on the clean test set (saved by notebook 04).
Goal: find out *why* the model fails, not just how often.

In [1]:
import sys
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")
from preprocess import LABELS
from metrics import get_scores, print_report

pd.set_option("display.max_colwidth", 200)
preds = pd.read_csv("../results/best_predictions_test_clean.csv")
train = pd.read_csv("../data/processed/train.csv")
preds["correct"] = preds["label"] == preds["pred"]
print_report(preds["label"], preds["pred"])

              precision    recall  f1-score   support

    negative      0.705     0.818     0.757      1708
     neutral      0.000     0.000     0.000       429
    positive      0.582     0.665     0.620      1091

    accuracy                          0.657      3228
   macro avg      0.429     0.494     0.459      3228
weighted avg      0.570     0.657     0.610      3228



## Which mistakes happen most

In [2]:
errors = preds[~preds["correct"]]
print(f"{len(errors)} errors out of {len(preds)} ({len(errors) / len(preds):.1%})")
errors.groupby(["label", "pred"]).size().sort_values(ascending=False).rename("count").to_frame()

1106 errors out of 3228 (34.3%)


count
label    pred           
positive negative    366
negative positive    311
neutral  negative    219
         positive    210

## Error type 1: unknown words

Do tweets with more words never seen in training get more errors? Spelling variation and new slang
would show up here.

In [3]:
train_words = set(" ".join(train["text"]).split())


def unseen_ratio(text):
    words = text.split()
    return sum(w not in train_words for w in words) / max(len(words), 1)


preds["unseen_ratio"] = preds["text"].apply(unseen_ratio)
preds["unseen_bin"] = pd.cut(preds["unseen_ratio"], [-0.01, 0.05, 0.15, 0.3, 1.0],
                             labels=["0-5%", "5-15%", "15-30%", "30%+"])
by_unseen = preds.groupby("unseen_bin", observed=True).agg(tweets=("correct", "size"), accuracy=("correct", "mean"))
by_unseen["macro_f1"] = [get_scores(g["label"], g["pred"])["macro_f1"]
                         for _, g in preds.groupby("unseen_bin", observed=True)]
by_unseen.round(3)

,tweets,accuracy,macro_f1
unseen_bin,,,
0-5%,1884,0.654,0.461
5-15%,958,0.689,0.471
15-30%,335,0.612,0.428
30%+,51,0.490,0.344


## Error type 2: tweet length

In [4]:
preds["n_words"] = preds["text"].str.split().str.len()
preds["len_bin"] = pd.cut(preds["n_words"], [0, 8, 15, 25, 100], labels=["1-8", "9-15", "16-25", "26+"])
preds.groupby("len_bin", observed=True)["correct"].agg(["size", "mean"]).round(3)

,size,mean
len_bin,,
1-8,734,0.515
9-15,1010,0.631
16-25,835,0.717
26+,649,0.783


## Error type 3: English vs Pidgin heavy tweets

In [5]:
# a small list of very common pidgin function words, used as a rough "how pidgin is this tweet" score
PIDGIN_WORDS = {"dey", "na", "wey", "una", "dem", "don", "wetin", "abeg", "sef", "sha", "oya", "comot",
                "pikin", "wahala", "sabi", "chop", "am", "e", "go", "make", "no", "fit", "say", "ehn", "o"}


def pidgin_ratio(text):
    words = text.split()
    return sum(w in PIDGIN_WORDS for w in words) / max(len(words), 1)


preds["pidgin_ratio"] = preds["text"].apply(pidgin_ratio)
preds["pidgin_bin"] = pd.cut(preds["pidgin_ratio"], [-0.01, 0.05, 0.15, 0.3, 1.0],
                             labels=["mostly english", "light pidgin", "pidgin", "heavy pidgin"])
preds.groupby("pidgin_bin", observed=True)["correct"].agg(["size", "mean"]).round(3)

,size,mean
pidgin_bin,,
mostly english,103,0.767
light pidgin,687,0.751
pidgin,1468,0.660
heavy pidgin,970,0.575


## Examples to read

I read these myself and sort them into reasons (sarcasm, mixed feelings, label I disagree with,
slang, neutral confused with negative, etc.).

In [6]:
print("=== examples the model got right ===")
preds[preds["correct"]].sample(8, random_state=1)[["text", "label", "pred"]]

=== examples the model got right ===


,text,label,pred
2097,hmmn bros come visit me with that automobile for my area for one month the car go turn motoodey nor dey tell person say sun hot,negative,negative
667,me sef dey find am,positive,positive
2404,na person like you go de lock window when your neighbor children gather to watch tv through window those days oga i de here continue the lamentations,negative,negative
2458,omo e de shake my bones laidis oo,negative,negative
2112,dem no wan hear word na sweet in the middle kill dem der,negative,negative
529,abeg incase una see masquerade wey dey over dance na me oo i dey hustle my transport back to abuja,positive,positive
676,wetin i wan take beg you to followback na gtgtbeg me,positive,positive
2309,wicked after u don drink people hennessy finish u no want make dem come drink ur own lai lai e no go happen,negative,negative


In [7]:
print("=== neutral tweets (true label) the model got wrong ===")
errors[errors["label"] == "neutral"].sample(10, random_state=1)[["text", "label", "pred"]]

=== neutral tweets (true label) the model got wrong ===


,text,label,pred
1318,oga u don dey port,neutral,positive
1299,i get work wey i dey do that side wan prime on tuesday then run asphalt next week,neutral,positive
1255,word with jennifer omoko weti bird see chop nai e take dey hammer or nai e go take hammer weti you understand by this proverb,neutral,negative
1368,nor be una dey find proof,neutral,negative
1153,i still dey reason wetin better days dey do ahead,neutral,positive
1263,mk i dey look you go na aisha dey hold my hand,neutral,positive
1419,na we dey rush our phone,neutral,positive
1095,nah we were both in a hurry it was yrs ago sef i remembercus it was a day to oshuns wedding went to pick up my suit den,neutral,positive
1333,na the same anambra wey i dey o,neutral,positive
1474,for this one wey una don talk so i dey resume tomorrow abi evening session dey today,neutral,positive


In [8]:
print("=== positive predicted as negative ===")
errors[(errors["label"] == "positive") & (errors["pred"] == "negative")].sample(10, random_state=1)[["text", "label", "pred"]]

=== positive predicted as negative ===


,text,label,pred
753,you serious forget awon seniors too bad na small remain make i propose to you for hod office when una wan graduate,positive,negative
375,true thing sha e dey better to kpim for this kain matters,positive,negative
703,na heaven gate we dey be that o but if any person fit try convert us we fit still reason am sha,positive,negative
480,lots of things don dey change for life pls tag someone u know has don this information is power,positive,negative
470,mehn the cars eeh fada lawd abeg nau me sef wan get car and house for assokoro,positive,negative
881,this is so funny to me because atleast una own dey form ball my home made puff puff dey always turn to fried egg because its always flat i cant even vex please teach me how to make proper puff puff,positive,negative
628,you still dey wait for my failureyou go wait for eternity mr haters god say our blessings na forever say go fall for our side but e no go reach us,positive,negative
424,no use laugh kill pesin abeg,positive,negative
821,e be like say na like this we go settle am patronize me i dey try raise tuition fee tuition fee don increase nigeria,positive,negative
10,breaking news the matter wey dey our table now be say aka starboy don talk say him go die for the matter one go let settle so simi zlatan adekunle,positive,negative


In [9]:
print("=== negative predicted as positive ===")
errors[(errors["label"] == "negative") & (errors["pred"] == "positive")].sample(10, random_state=1)[["text", "label", "pred"]]

=== negative predicted as positive ===


,text,label,pred
2034,even u dey shout nite on and na k subs u dey do always na wa oo,negative,positive
2624,baba you body na blessing i used to be like that o but oja ti burst and head self don dey fold,negative,positive
3207,which kind of half don corper am i like this self them no gree make i enjoy camp for all the fine girls don forget my face self from batch a stream to batch c congrats on your compulsory vacation ...,negative,positive
2568,wahala for who dey use cutlery chop swallow,negative,positive
2012,mama you dey offline since na d gbege of all of us dey since o,negative,positive
2398,is over btween us oo wicked girl u nver loved me bfor sef,negative,positive
3222,na wa oo dem dey watch porn for afternoon,negative,positive
2770,please lets greet her her boyfriend na sars na her account those pos money dey enter,negative,positive
2340,thread n replies under this thread sef dey one kain,negative,positive
2657,if they wan hack na video tutorial you go see,negative,positive


## My manual categories

I go through a sample of 50 errors and put each one in a category. (Filled in by hand, see the table
below.)

In [10]:
sample = errors.sample(50, random_state=7)[["text", "label", "pred"]].reset_index(drop=True)
sample.to_csv("../results/error_sample_to_label.csv", index=False)
sample

,text,label,pred
0,na weyrey wey dey disguise,negative,positive
1,you sef why did u gbe idi e si odo ee,negative,positive
2,the energy wey i put for love if i use am learn mechanic i for don sabi repair train,negative,positive
3,i dey there when kanu do he heart operation na me hold he phone,positive,negative
4,na man wey dey reason dey scratch bear bear,neutral,negative
5,my guy dey de list o,neutral,positive
6,when be say make u read u wont na instablog u go dey go read headline,negative,positive
7,wetin i do,neutral,positive
8,by the time beans done who cook indomie go don dey hungry,positive,negative
9,cubana chiefpriest na who blow kon help person blow na igbo people dey call odogwu me david omo adeleke aka obo na the real odogwu,positive,negative
